In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import warnings
warnings.filterwarnings('ignore')
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

# SQLDB

SQLDB 是基于关系数据库构建的因子库

基于关系数据库构建的因子库和数据库原始对象的对应关系：
* 整个数据库对应于因子库
* 每张数据库表对应于因子表
* 每张数据库表的字段对应于单个因子

本质上是将一个二维的因子数据矩阵挤压成具有二重索引的一维向量. 对于因子数据的访问, 内部使用标准的 SQL 查询语句完成.

SQLDB 支持多种因子表类型，以适配不同的数据组织方式：

| 表类型 | 说明 | 适用场景 |
|--------|------|----------|
| **WideTable**（宽表） | 每行由一个 (时点, ID) 唯一确定，每列是一个因子 | 日频行情、财务指标等标准面板数据 |
| **NarrowTable**（窄表） | 因子名列和因子值列分别存储在两个字段中 | 灵活 schema 的数据，如事件数据 |
| **FeatureTable**（特征表） | 无时点维度，只有 ID 维度 | 静态属性，如股票上市板块 |
| **TimeSeriesTable**（时序表） | 无 ID 维度，只有时点维度 | 宏观指标，如利率、汇率 |
| **MappingTable**（映射表） | 存储 ID 之间的映射关系 | 代码对照表、成分股映射 |

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库]
        B1[因子表]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子1]
        F2[因子2]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[关系数据库]
        D1[数据库表]
        D2[数据库表]
        C --> D1
        C --> D2
        E2[字段2]
        E1[字段1]
        D2 --> E1
        D2 --> E2
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E1

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
```

以下代码要求有可以访问的 postgresql 数据库，且设置好了配置文件。或者执行 [import_postgres_sqldb_demo_data.py](../tools/import_postgres_sqldb_demo_data.py) 脚本生成示例数据，这要求有写入权限的 postgresql 数据库。

SQLDB 的配置文件默认位于用户目录下的 “QuantStudioConfig” 文件夹里的 "SQLDBConfig.json" 文件。通常将数据库的连接信息配置到文件里，示例如下:
```json
{
    "Name": "SQLDB",
    "DBType": "PostgreSQL",
    "DBName": "QSData",
    "IPAddr": "localhost",
    "Port": 5432,
    "User": "postgres",
    "Pwd": "123456",
    "TablePrefix": "",
    "CharSet": "utf8",
    "Connector": "default",
    "IDField": "code",
    "DTField": "datetime"
}
```

主要配置参数说明：

| 参数 | 类型 | 说明 |
|------|------|------|
| `Name` | str | 因子库名称，默认 `"SQLDB"` |
| `DBType` | Literal | 数据库类型：`"MySQL"` / `"SQL Server"` / `"Oracle"` / `"PostgreSQL"` |
| `DBName` | str | 数据库名 |
| `IPAddr` | str | 数据库 IP 地址 |
| `Port` | int | 数据库端口 |
| `User` | str | 用户名 |
| `Pwd` | str | 密码 |
| `CharSet` | str | 字符集，如 `"utf8"` |
| `Connector` | str | 连接器：`"default"` / `"psycopg2"` / `"pymysql"` 等 |
| `TablePrefix` | str | 数据库表名前缀 |
| `InnerPrefix` | str | 内部前缀，默认 `"qs_"`，SQLDB 只识别该前缀的表 |
| `DTField` | str | 时点字段名，默认 `"datetime"` |
| `IDField` | str | ID 字段名，默认 `"code"` |
| `IgnoreFields` | List[str] | 忽略的字段名列表 |
| `CheckWriteData` | bool | 写入时是否校验数据一致性，默认 `False` |
| `CheckNullable` | bool | 写入时是否检查 NULL 约束，默认 `False` |
| `MetaTableName` | str | 元数据侧表名，默认 `"qs_meta"` |

In [3]:
# 创建因子库对象并 connect
from QuantStudio.Factor.SQLDB import SQLDB

FDB = SQLDB().connect()
print(qs_help(FDB))

类型: SQLDB
模块: QuantStudio.Factor.SQLDB
QS 对象类型: 因子库
QS 对象名称: SQLDB
QSID: b1f0b13971222d974a92c0f516502c92260f6d3bc58925b2c19ee3e5912945ee
参数集:
    * Name(名称): <class 'str'>, 默认值 'SQLDB', 当前取值: 'SQLDB'
    * DBType(数据库类型): typing.Literal['MySQL', 'SQL Server', 'Oracle', 'PostgreSQL'], 默认值 'MySQL', 当前取值: 'PostgreSQL'
    * DBName(数据库名): <class 'str'>, 默认值 'Scorpion', 当前取值: 'KDB_dev'
    * IPAddr(IP地址): <class 'str'>, 默认值 '127.0.0.1', 当前取值: 'localhost'
    * Port(端口): <class 'int'>, 默认值 3306, 当前取值: 5432
    * User(用户名): <class 'str'>, 默认值 'root', 当前取值: 'shzq'
    * TablePrefix(表名前缀): <class 'str'>, 默认值 '', 当前取值: ''
    * CharSet(字符集): typing.Literal['utf8', 'utf8mb4', 'gbk', 'gb2312', 'gb18030', 'cp936', 'big5'], 默认值 'utf8', 当前取值: 'utf8'
    * Connector(连接器): typing.Literal['default', 'cx_Oracle', 'pymssql', 'mysql.connector', 'pymysql', 'psycopg2', 'pyodbc'], 默认值 'default', 当前取值: 'default'
    * ConnRetryNum(连接重试次数): <class 'int'>, 默认值 3, 当前取值: 3
    * ConnIntervalSeconds(连接重试间隔): <class

In [4]:
# 获取因子库中的因子表列表
print(FDB.TableNames[:5])

['_industry_cn_report', '_qs_meta', '_stock_cn_report', 'index_cn_day_bar', 'macro_indicator']


# 因子表

In [5]:
# 获取因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
print(qs_help(FT))

类型: SQL_WideTable
模块: QuantStudio.Factor.FactorUtils
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_cn_day_bar
QSID: 6781381763a4f0a70531a88058b70feb6b59ec8de59e0bf502bce53ddf7106d3
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'stock_cn_day_bar'
    * TableType(因子表类型): typing.Literal['WideTable'], 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'WideTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: 'datetime'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: 0
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 appl

In [6]:
# 因子列表
FT = FDB.getTable("stock_cn_day_bar")
print(FT.FactorNames)

['amount', 'close', 'code', 'datetime', 'high', 'low', 'open', 'volume']


## 表元信息

In [7]:
# 获取因子表元信息的方法说明
print(qs_help(FT.getMetaData))

类型: method (bound to SQL_WideTable)
模块: QuantStudio.Factor.FactorUtils
签名: SQL_Table.getMetaData(key=None)
说明文档:
    获取因子表的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [8]:
# 获取因子表的所有元信息
print(FT.getMetaData())

DBTableName    qs_stock_cn_day_bar
Description                   None
TableClass               WideTable
Name: stock_cn_day_bar, dtype: object


## 时点序列

In [9]:
# 获取时点序列的方法
print(qs_help(FT.getDateTime))

类型: method (bound to SQL_WideTable)
模块: QuantStudio.Factor.FactorUtils
签名: SQL_WideTable.getDateTime(ifactor_name=None, iid=None, start_dt=None, end_dt=None)
说明文档:
    获取时点序列
    
    Args:
        ifactor_name: 给定的因子名称, 非 None 表示获取该因子的时点序列, None 表示获取表的时点序列
        iid: 给定的 ID, 非 None 表示获取该 ID 的时点序列, None 表示获取所有的时点序列
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该因子表没有固定的时点序列或者无法获取


In [10]:
# 给定起始时点和截止时点, 获取因子表的时点序列
FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 5))

[datetime.datetime(2025, 1, 1, 0, 0),
 datetime.datetime(2025, 1, 2, 0, 0),
 datetime.datetime(2025, 1, 3, 0, 0),
 datetime.datetime(2025, 1, 4, 0, 0),
 datetime.datetime(2025, 1, 5, 0, 0)]

## ID 序列

In [11]:
# 获取因子表 ID 序列的方法
print(qs_help(FT.getID))

类型: method (bound to SQL_WideTable)
模块: QuantStudio.Factor.FactorUtils
签名: SQL_WideTable.getID(ifactor_name=None, idt=None)
说明文档:
    获取 ID 序列
    
    Args:
        ifactor_name: 给定的因子名称, 非 None 表示获取该因子的 ID 序列, None 表示获取表的 ID 序列
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该因子表没有固定的 ID 序列或者无法获取


In [12]:
# 获取因子表的 ID 序列
IDs = FT.getID()
print(IDs[:5], "..." if len(IDs)>5 else "", f"共 {len(IDs)} 个")

['000001.SZ', '000002.SZ', '000003.SZ', '000004.SZ', '000005.SZ'] ... 共 20 个


## 读取数据

In [13]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
Data = FT.readData(factor_names=["close", "open"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

因子表数据
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: close to open
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


## 其他因子表类型

除了默认的 WideTable 外，SQLDB 还支持多种因子表类型，通过 `getTable` 的 `TableType` 参数指定。

### NarrowTable（窄表）

窄表将因子名和因子值分别存储在两个字段中，而不是每个因子占一列。适用于字段不固定的场景。

窄表需要指定：
- `FactorNameField`：存储因子名的字段
- `FactorValueField`：存储因子值的字段（默认取第一个非维度字段）
- `IDField` / `DTField`：ID 和时点字段

In [14]:
# 窄表示例：stock_cn_factor_value_narrow 表
# 窄表需要指定 FactorNameField（因子名字段）和 FactorValueField（因子值字段）
NarrowFT = FDB.getTable("stock_cn_factor_value_narrow", args={
    "TableType": "NarrowTable",
    "FactorNameField": "factor_name",
    "FactorValueField": "factor_value",
    "LookBack": 0
})
print(qs_help(NarrowFT))
print(f"\n因子列表: {NarrowFT.FactorNames}")

# 读取窄表数据
Data = NarrowFT.readData(factor_names=["pe", "pb"], ids=IDs, dts=DTs[:3])
print(f"\n窄表读取结果:\n{Data}")

类型: SQL_NarrowTable
模块: QuantStudio.Factor.FactorUtils
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_cn_factor_value_narrow
QSID: a731bfd9b515d8feb97d5766488be475f14358e7c7759a101c9a8a279a68d3e0
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'stock_cn_factor_value_narrow'
    * TableType(因子表类型): typing.Literal['NarrowTable'], 默认值 'NarrowTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'NarrowTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: 'datetime'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: 0
    * FactorNameField(因子名字段): <class 'str'>, 无默认值, 指示因子名称的字段, 默认值内部自动判断, 当前取值: 'factor_name'
    * FactorValueField(因子值字段): <class 'str'>, 无默认值, 指示因子取值的字段, 默认值内部自动判断, 当前取值: 'factor_value'
    * MultiMapping(多重映射): <class 'bool'>, 默认值 True, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: True
    * Operator(算子): ty

### FeatureTable（特征表）

特征表没有时点维度，只有 ID 维度。适用于存储证券的静态属性，如行业分类、上市板块、公司全称等。

当 SQLDB 检测到表中没有时点字段时，会自动将表类型设为 `FeatureTable`。

In [15]:
# 特征表示例：stock_cn_static_info 表存储了股票静态属性
FeatureFT = FDB.getTable("stock_cn_static_info", args={"TableType": "FeatureTable"})
print(qs_help(FeatureFT))
print(f"因子列表: {FeatureFT.FactorNames}")

# 读取特征数据，dts 参数必须传但特征表会使用最大时点的数据
IDs = ["000001.SZ", "000002.SZ"]
Data = FeatureFT.readData(factor_names=["full_name", "listed_date"], ids=IDs, dts=[dt.datetime(2025, 1, 1)])
print("\n特征表数据:")
print(Data)

类型: SQL_FeatureTable
模块: QuantStudio.Factor.FactorUtils
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_cn_static_info
QSID: 6cb3f8f5f119d08d1ef189100830e9f417513bb2a690e0683bb4ff9023f48e59
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'stock_cn_static_info'
    * TableType(因子表类型): typing.Literal['FeatureTable'], 默认值 'FeatureTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'FeatureTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: 'datetime'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 inf, 当前取值: inf
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 ap

### TimeSeriesTable（时序表）

时序表没有 ID 维度，只有时点维度。适用于存储宏观指标，如无风险利率、汇率、GDP 等。

当 SQLDB 检测到表中没有 ID 字段时，会自动将表类型设为 `TimeSeriesTable`。

### MappingTable（映射表）

映射表用于存储 ID 之间的映射关系，如成分股与指数的从属关系。通常包含 `StartDT` 和 `EndDT` 字段标识映射的有效期。

In [16]:
# 时序表示例：macro_indicator 存储宏观指标，无 ID 维度
TSFT = FDB.getTable("macro_indicator", args={"TableType": "TimeSeriesTable"})
print(qs_help(TSFT))
print(f"因子列表: {TSFT.FactorNames}")

# 读取时序数据，ids 参数传空列表
MacroDTs = [dt.datetime(2020, 1, 1), dt.datetime(2020, 4, 1), dt.datetime(2020, 7, 1)]
Data = TSFT.readData(factor_names=["interest_rate", "cpi"], ids=[], dts=MacroDTs)
print(f"\n时序表数据:\n{Data}")

类型: SQL_TimeSeriesTable
模块: QuantStudio.Factor.FactorUtils
QS 对象类型: 计算节点-因子表
QS 对象名称: macro_indicator
QSID: 2383c54fb727015453a41b948ef299bdfd0e329562f4b6412991dc9e4a44d215
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'macro_indicator'
    * TableType(因子表类型): typing.Literal['TimeSeriesTable'], 默认值 'TimeSeriesTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'TimeSeriesTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: 'datetime'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 inf, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: inf
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 apply 的函数 f(x), 其中 x 为 Series, 默认值 None 表示使用 lambda x: x.tolist(), 当前取值: None
    * OperatorDataType(算子数据类型): typing.Literal['object'

In [17]:
# 映射表示例：stock_index_mapping 存储股票-指数映射关系
# 映射表 ID 字段名不是默认的 code，需要显式指定 IDField
MapFT = FDB.getTable("stock_index_mapping", args={
    "TableType": "MappingTable",
    "IDField": "stock_code"
})
print(qs_help(MapFT))
print(f"因子列表: {MapFT.FactorNames}")

# 读取映射数据
Data = MapFT.readData(factor_names=["index_code", "start_date", "end_date"],
                       ids=["000001.SZ"], dts=[dt.datetime(2025, 1, 1)])
print(f"\n映射表数据:\n{Data}")

类型: SQL_MappingTable
模块: QuantStudio.Factor.FactorUtils
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_index_mapping
QSID: b469576f23fade3cf4f02d2e438d5a9fd4e83b81fcc547692eb639d90a3ae48d
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'stock_index_mapping'
    * TableType(因子表类型): typing.Literal['MappingTable'], 默认值 'MappingTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'MappingTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: 'datetime'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即起始时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: True
    * EndDTField(结束时点字段): <class 'str'>, 无默认值, 用以指示结束填充的时点字段, 默认值 None 表示内部自动判断, 当前取值: 'datetime'
    * EndDTIncluded(包含结束时点): <class 'bool'>, 默认值 True, 结束时点处是否填充数据, 当前取值: False
说明文档:
    基于 SQL 数据库表的映射因子表
    一个字段（参数IDField指定）标识 ID, 一个字段（参数DTField指定）标识起始时点, 一个字段（参数EndDTField指定）标识截止时点, 其余字段为因子
因子列表: ['dat

# 因子

In [18]:
# 获取因子对象
F = FT.getFactor("close")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: close
QSID: 28982d16234c25632a97f6c34df2eae4a35db1236185a23969a935950b725eeb
参数集:
    * Name(名称): <class 'str'>, 默认值 'Factor', 当前取值: 'close'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
说明文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


## 因子元信息

In [19]:
# 获取因子元信息的方法
print(qs_help(F.getMetaData))

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getMetaData(key: Optional[str] = None) -> Union[Any, pandas.core.series.Series]
说明文档:
    获取因子的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [20]:
# 获取因子的所有元信息
print(F.getMetaData(key=None))

# 获取因子的数据类型
print(f"\n数据类型: {F.getMetaData(key='DataType')}")

DBFieldName          close
DataType            double
Description           None
FieldKey                  
FieldType               因子
Nullable               YES
Supplementary         None
TableDescription      None
dtype: object

数据类型: double


## 因子时点序列

In [21]:
# 获取因子时点序列的方法
print(qs_help(F.getDateTime))

# 获取因子的时点序列
print(f"\n时点序列: {F.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 5))}")

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getDateTime(iid: Optional[str] = None, start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None, **kwargs) -> List[datetime.datetime]
说明文档:
    获取时点序列
    
    Args:
        iid: 给定的 ID, 非 None 表示获取该 ID 的时点序列, None 表示获取所有的时点序列
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该因子没有固定的时点序列或者无法获取

时点序列: [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0), datetime.datetime(2025, 1, 4, 0, 0), datetime.datetime(2025, 1, 5, 0, 0)]


## 因子 ID 序列

In [22]:
# 获取因子 ID 序列的方法
print(qs_help(F.getID))

# 获取因子的 ID 序列
IDs = F.getID()
print(f"\nID 序列: {IDs[:5]}... 共 {len(IDs)} 个")

类型: method (bound to Factor)
模块: QuantStudio.Factor.Factor
签名: Factor.getID(idt: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    获取 ID 序列
    
    Args:
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该因子没有固定的 ID 序列或者无法获取

ID 序列: ['000001.SZ', '000002.SZ', '000003.SZ', '000004.SZ', '000005.SZ']... 共 20 个


## 读取数据

In [23]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

                     000001.SZ  000002.SZ
2025-01-01 00:00:00   8.115185   4.760840
2025-01-02 00:00:00   0.352198   1.806606
2025-01-03 00:00:00   6.019437   0.633690
2025-01-04 00:00:00   3.936297   3.755494
2025-01-05 00:00:00   4.884425   1.342670

# 因子库的变更操作

SQLDB 是支持写入和变更的因子库（继承自 `QuantStudio.Factor.FactorDB.WritableFactorDB`）。以下演示各种变更操作。

## 数据写入

`writeData` 方法将 Panel 数据写入因子库。`if_exists` 参数控制写入模式：

| 模式 | 说明 |
|------|------|
| `"update"` | 用新数据更新原数据（默认），新数据覆盖旧值，保留未涉及的因子 |
| `"replace"` | 完全用新数据替换原数据，未涉及的因子从旧数据中取回 |
| `"append"` | 不覆盖原有数据，只在缺失位置填充新数据 |
| `"update_notnull"` | 仅用非空的新数据覆盖，空值保留原数据 |

In [24]:
# 因子库数据写入方法
print(qs_help(FDB.writeData))

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.writeData(data: QuantStudio.Core.QSObject.Panel, table_name: str, if_exists: Literal['update', 'replace', 'append'] = 'update', data_type: Dict[str, Literal['double', 'string', 'object']] = {}, **kwargs)
说明文档:
    写入数据
    
    Args:
        data: 待写入的因子数据, Panel(items=[因子], major_axis=[时点], minor_axis=[ID])
        table_name: 因子表名称
        if_exists: 如果该因子已经存在时数据写入的方式, update 表示用新数据更新原数据, append 表示不更新原数据而只增加原来没有的数据, replace 表示完全用新数据替换原数据, 等同于先删除原数据再写入
        data_type: 待写入因子的数据类型, {因子名称: "double" or "string" or "object"}, 如果 data_type 未指定某个因子的数据类型，则交由系统判定


In [25]:
# 数据写入示例
from QuantStudio.Core.QSObject import Panel

IDs = [str(i).zfill(6) + ".SZ" for i in range(1, 4)]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]

Data = Panel({
    "Factor1": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs),
    "Factor2": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs)
})
print("待写入的数据 : ")
print(Data)

print("-" * 60)
TargetTable = "TestTable"
FDB.writeData(data=Data, table_name=TargetTable, if_exists="update", data_type={"Factor1":"double", "Factor2":"double"})
print("写入后的因子表列表 : ")
print(FDB.TableNames)

print("-" * 60)
print("写入后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

待写入的数据 : 
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 3 (minor_axis)
Items axis: Factor1 to Factor2
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000003.SZ
------------------------------------------------------------
写入后的因子表列表 : 
['TestTable', '_industry_cn_report', '_qs_meta', '_stock_cn_report', 'index_cn_day_bar', 'macro_indicator', 'qs_meta', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_factor_value_narrow', 'stock_cn_industry', 'stock_cn_static_info', 'stock_cn_status', 'stock_index_mapping']
------------------------------------------------------------
写入后的因子列表 : 
['Factor1', 'Factor2', 'code', 'datetime']


## 创建表

`createTable` 方法用于在因子库中创建一张新的因子表。

In [26]:
# 创建表的方法
print(qs_help(FDB.createTable))

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.createTable(table_name: str, field_types: Dict[str, str])
说明文档:
    ⚠️  未找到文档字符串（包括父类）
    该对象可能：
    - 是内置函数/方法（C 实现，无 __doc__）
    - 确实没有文档


## 添加因子

`addFactor` 方法用于向已有表中添加新的因子字段。如果表不存在，会自动创建该表。

In [27]:
# 添加因子的方法
print(qs_help(FDB.addFactor))

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.addFactor(table_name: str, field_types: Dict[str, str])
说明文档:
    添加因子
    
    Args:
        table_name: 表名
        field_types: {字段名: 数据库数据类型}


## 设置表的元信息

In [28]:
# 设置因子表元信息的方法
print(qs_help(FDB.setTableMetaData))

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.setTableMetaData(table_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
说明文档:
    设置因子表的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 因子表名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [29]:
# 设置表的元信息
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(FT.getMetaData())

print("-" * 60)
FDB.setTableMetaData(table_name=TargetTable, key="Description", value="这是一张测试表")
print("设置后的元信息 : ")
print(FT.getMetaData())

设置前的元信息 : 
DBTableName    qs_TestTable
Description             NaN
TableClass        WideTable
Name: TestTable, dtype: object
------------------------------------------------------------
设置后的元信息 : 
DBTableName    qs_TestTable
Description             NaN
TableClass        WideTable
Name: TestTable, dtype: object


## 设置因子的元信息

In [30]:
# 设置因子元信息的方法
print(qs_help(FDB.setFactorMetaData))

# 设置因子的元信息
TargetTable = "TestTable"
TargetFactor = "Factor1"
FT = FDB.getTable(TargetTable)
print("\n设置前的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

print("-" * 60)
FDB.setFactorMetaData(table_name=TargetTable, ifactor_name=TargetFactor, key="Description", value="这是一个测试因子")
print("设置后的元信息 : ")
print(FT.getFactorMetaData(factor_names=[TargetFactor]))

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.setFactorMetaData(table_name: str, ifactor_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
说明文档:
    设置因子的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 因子表名称
        ifactor_name: 因子名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息

设置前的元信息 : 
          DBFieldName DataType Nullable FieldKey Description TableDescription  \
FieldName                                                                       
Factor1       Factor1   double      NaN      NaN         NaN              NaN   

          FieldType Supplementary  
FieldName                          
Factor1          因子          None  
------------------------------------------------------------
设置后的元信息 : 
          DBFieldName DataType Nullable FieldKey Description TableDescription  \
FieldName                                                                       
Factor1       Factor1   double   

## 重命名因子

In [31]:
# 重命名因子的方法
print(qs_help(FDB.renameFactor))

# 重命名因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print(f"\n重命名前因子列表 : {FT.FactorNames}")

FDB.renameFactor(table_name=TargetTable, old_factor_name="Factor1", new_factor_name="NewFactor1")
print(f"重命名后因子列表 : {FT.FactorNames}")

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.renameFactor(table_name: str, old_factor_name: str, new_factor_name: str)
说明文档:
    重命名因子

重命名前因子列表 : ['Factor1', 'Factor2', 'code', 'datetime']
重命名后因子列表 : ['Factor1', 'Factor2', 'code', 'datetime']


## 删除因子

In [32]:
# 删除因子的方法
print(qs_help(FDB.deleteFactor))

# 删除因子
TargetTable = "TestTable"
FT = FDB.getTable(TargetTable)
print(f"\n删除前因子列表 : {FT.FactorNames}")

FDB.deleteFactor(table_name=TargetTable, factor_names=["NewFactor1"])
print(f"删除后因子列表 : {FT.FactorNames}")

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.deleteFactor(table_name: str, factor_names: List[str])
说明文档:
    删除给定表中的某些因子
    
    Args:
        table_name: 因子表名称
        factor_names: 待删除的因子名列表

删除前因子列表 : ['NewFactor1', 'Factor2', 'code', 'datetime']
删除后因子列表 : ['NewFactor1', 'Factor2', 'code', 'datetime']


## 重命名表

In [33]:
# 重命名表的方法
print(qs_help(FDB.renameTable))

# 重命名表
print(f"重命名前因子表 : {FDB.TableNames}")

FDB.renameTable(old_table_name="TestTable", new_table_name="NewTestTable")
print(f"重命名后因子表 : {FDB.TableNames}")

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.renameTable(old_table_name: str, new_table_name: str)
说明文档:
    重命名表
    
    Args:
        old_table_name: 原表名
        new_table_name: 新表名
重命名前因子表 : ['TestTable', '_industry_cn_report', '_qs_meta', '_stock_cn_report', 'index_cn_day_bar', 'macro_indicator', 'qs_meta', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_factor_value_narrow', 'stock_cn_industry', 'stock_cn_static_info', 'stock_cn_status', 'stock_index_mapping']
重命名后因子表 : ['NewTestTable', '_industry_cn_report', '_qs_meta', '_stock_cn_report', 'index_cn_day_bar', 'macro_indicator', 'qs_meta', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_factor_value_narrow', 'stock_cn_industry', 'stock_cn_static_info', 'stock_cn_status', 'stock_index_mapping']


## 删除表

In [34]:
# 删除表的方法
print(qs_help(FDB.deleteTable))

# 删除表
print(f"删除前因子表 : {FDB.TableNames}")

FDB.deleteTable(table_name="NewTestTable")
print(f"删除后因子表 : {FDB.TableNames}")

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.deleteTable(table_name: str)
说明文档:
    删除表
    
    Args:
        table_name: 表名
删除前因子表 : ['NewTestTable', '_industry_cn_report', '_qs_meta', '_stock_cn_report', 'index_cn_day_bar', 'macro_indicator', 'qs_meta', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_factor_value_narrow', 'stock_cn_industry', 'stock_cn_static_info', 'stock_cn_status', 'stock_index_mapping']
删除后因子表 : ['_industry_cn_report', '_qs_meta', '_stock_cn_report', 'index_cn_day_bar', 'macro_indicator', 'qs_meta', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_factor_value_narrow', 'stock_cn_industry', 'stock_cn_static_info', 'stock_cn_status', 'stock_index_mapping']


## 删除数据

`deleteData` 方法用于删除表中指定时点或指定 ID 的数据。可以通过 `ids`、`dts` 或 `dt_ids`（时点+ID 组合）来精确控制删除范围。

In [35]:
# 删除数据的方法
print(qs_help(FDB.deleteData))

类型: method (bound to SQLDB)
模块: QuantStudio.Factor.SQLDB
签名: SQLDB.deleteData(table_name: str, ids: Optional[List[str]] = None, dts: Optional[List[datetime.datetime]] = None, dt_ids: Optional[List[Tuple[datetime.datetime, str]]] = None)
说明文档:
    ⚠️  未找到文档字符串（包括父类）
    该对象可能：
    - 是内置函数/方法（C 实现，无 __doc__）
    - 确实没有文档


# 元数据侧表

SQLDB 在数据库中维护了一张名为 `qs_meta` 的元数据侧表（可通过 `MetaTableName` 参数自定义表名），用于持久化存储因子表和因子的自定义元信息。

**侧表结构：**

| 列名 | 说明 |
|------|------|
| `table_name` | 因子表名称 |
| `field_name` | 因子名称（空字符串表示表级元数据） |
| `meta_key` | 元信息键 |
| `meta_value` | 元信息值 |

**设计要点：**

- 表级元数据：`field_name` 为空字符串，通过 `setTableMetaData` 写入
- 因子级元数据：`field_name` 为因子名，通过 `setFactorMetaData` 写入
- **级联更新**：重命名表/因子或删除表/因子时，侧表中的相关记录会自动同步更新或删除
- 侧表在 `connect()` 时自动创建（如果不存在），在 `connect()` 时自动加载元数据到 `_TableInfo` 和 `_FactorInfo`

元数据侧表使得 SQLDB 可以存储数据库 schema 之外的扩展信息（如因子描述、数据来源、更新频率等），而无需修改数据库表结构。